In [0]:
%pip install folium
%restart_python 

In [0]:
# Importações iniciais
def get_table_sample(table_path, n=100):
    return spark.read.table(table_path).limit(n)

flights_sample = get_table_sample('workspace.postech_tech_challenge_f3.flights')
airlines_sample = get_table_sample('workspace.postech_tech_challenge_f3.airlines')
airports_sample = get_table_sample('workspace.postech_tech_challenge_f3.airports')

# Exibir amostras dos datasets
print('Amostra de voos:')
display(flights_sample)

print('Amostra de companhias aéreas:')
display(airlines_sample)

print('Amostra de aeroportos:')
display(airports_sample)

In [0]:
# Pipeline de clusterização de rotas
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
import matplotlib.pyplot as plt

# Featurização para clusterizar rotas
# Agrupa por origem-destino, calcula atraso médio e distância média
rotas_df = (
    spark.read.table('workspace.postech_tech_challenge_f3.flights')
    .groupBy('ORIGIN_AIRPORT', 'DESTINATION_AIRPORT')
    .agg(
        {'DEPARTURE_DELAY': 'avg', 'DISTANCE': 'avg'}
    )
    .withColumnRenamed('avg(DEPARTURE_DELAY)', 'avg_departure_delay')
    .withColumnRenamed('avg(DISTANCE)', 'avg_distance')
    .dropna()
    .limit(1000)
)

# Monta vetor de features
assembler = VectorAssembler(
    inputCols=['avg_departure_delay', 'avg_distance'],
    outputCol='features')
rotas_feat = assembler.transform(rotas_df)

# Normaliza os dados
scaler = StandardScaler(inputCol='features', outputCol='scaledFeatures')
scalerModel = scaler.fit(rotas_feat)
rotas_scaled = scalerModel.transform(rotas_feat)

# Clusterização
kmeans = KMeans(featuresCol='scaledFeatures', k=4, seed=42)
model = kmeans.fit(rotas_scaled)
rotas_clustered = model.transform(rotas_scaled)

# Converte para pandas para visualização
pd_rotas = rotas_clustered.select('ORIGIN_AIRPORT','DESTINATION_AIRPORT','avg_departure_delay', 'avg_distance', 'prediction').toPandas()

# Gráfico de dispersão: atraso médio x distância, colorido por cluster
plt.figure(figsize=(10,6))
for cluster in pd_rotas['prediction'].unique():
    cluster_data = pd_rotas[pd_rotas['prediction'] == cluster]
    plt.scatter(
        cluster_data['avg_distance'], 
        cluster_data['avg_departure_delay'], 
        label=f'Cluster {cluster}'
    )
plt.xlabel('Distância média da rota')
plt.ylabel('Atraso médio na partida (min)')
plt.title('Clusterização de rotas de voo por atraso e distância')
plt.legend()
plt.show()

# Interpretação inicial dos clusters
for cluster in sorted(pd_rotas['prediction'].unique()):
    descr = pd_rotas[pd_rotas['prediction'] == cluster][['avg_departure_delay','avg_distance']].describe()
    print(f'Cluster {cluster} - Resumo das rotas:')
    print(descr)

In [0]:
# Exibir rotas de um cluster específico
# Escolha o número do cluster de interesse (0, 1, 2, 3)
cluster_escolhido = 0  # Altere aqui para outro cluster se desejar

rotas_cluster = pd_rotas[pd_rotas['prediction'] == cluster_escolhido][[
    'ORIGIN_AIRPORT','DESTINATION_AIRPORT','avg_departure_delay','avg_distance']]
print(f'Rotas pertencentes ao cluster {cluster_escolhido}:')
display(rotas_cluster.sort_values('avg_departure_delay', ascending=False))

In [0]:
# Enriquecimento das rotas com coordenadas
# Une latitude e longitude do aeroporto de origem e destino

airports_pd = airports_sample.toPandas()  # já temos a amostra; use spark.read.table se precisar de mais

pd_rotas_coord = pd_rotas.merge(
    airports_pd[['IATA_CODE','LATITUDE','LONGITUDE']],
    left_on='ORIGIN_AIRPORT', right_on='IATA_CODE', how='left'
).rename(columns={'LATITUDE':'ORIGIN_LAT','LONGITUDE':'ORIGIN_LON'})
pd_rotas_coord = pd_rotas_coord.merge(
    airports_pd[['IATA_CODE','LATITUDE','LONGITUDE']],
    left_on='DESTINATION_AIRPORT', right_on='IATA_CODE', how='left'
).rename(columns={'LATITUDE':'DEST_LAT','LONGITUDE':'DEST_LON'})
pd_rotas_coord.drop(['IATA_CODE_x','IATA_CODE_y'], axis=1, inplace=True)

# Filtra linhas sem coordenadas para plotagem
pd_rotas_coord = pd_rotas_coord.dropna(subset=['ORIGIN_LAT','ORIGIN_LON','DEST_LAT','DEST_LON'])

# Exibe amostra
print(pd_rotas_coord.head())

In [0]:
# Mapa geográfico das rotas e atrasos
import folium
from folium import plugins

# Centro inicial do mapa (EUA central)
map_center = [39.8283, -98.5795]
rotas_map = folium.Map(location=map_center, zoom_start=4)

# Função para definir cor da rota pelo atraso
def delay_color(delay):
    if delay < 5:
        return 'green'
    elif delay < 15:
        return 'orange'
    else:
        return 'red'

# Plotar cada rota (limitando para 100)
for _, row in pd_rotas_coord.nlargest(100, 'avg_departure_delay').iterrows():
    folium.PolyLine(
        locations=[
            [row['ORIGIN_LAT'], row['ORIGIN_LON']],
            [row['DEST_LAT'], row['DEST_LON']]
        ],
        color=delay_color(row['avg_departure_delay']),
        weight=2,
        tooltip=f"{row['ORIGIN_AIRPORT']} - {row['DESTINATION_AIRPORT']}\nAtraso médio: {row['avg_departure_delay']:.2f} min"
    ).add_to(rotas_map)
    folium.CircleMarker(
        location=[row['ORIGIN_LAT'], row['ORIGIN_LON']],
        radius=3,
        color='blue',
        fill=True,
        fill_opacity=0.6,
        tooltip=row['ORIGIN_AIRPORT']
    ).add_to(rotas_map)
    folium.CircleMarker(
        location=[row['DEST_LAT'], row['DEST_LON']],
        radius=3,
        color='purple',
        fill=True,
        fill_opacity=0.6,
        tooltip=row['DESTINATION_AIRPORT']
    ).add_to(rotas_map)

rotas_map